In [4]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline

In [5]:
DATA_ROOT = Path(r"C:\Users\Sanjeev\OneDrive\Desktop\Brain ID\EEG\ds004148")
FEATURE_FILE = DATA_ROOT / "processed_windows" / "bandpower_features.csv"

features_df = pd.read_csv(FEATURE_FILE)

print("Shape:", features_df.shape)
print(features_df.head())

Shape: (8735, 18)
  subject       session  window  Fp1_delta  Fp1_theta  Fp1_alpha  Fp1_beta  \
0  sub-01  ses-session1       0   0.689025   0.080045   0.035803  0.061516   
1  sub-01  ses-session1       1   0.802907   0.054172   0.022807  0.033051   
2  sub-01  ses-session1       2   0.335598   0.153675   0.134009  0.109567   
3  sub-01  ses-session1       3   0.576905   0.081624   0.054894  0.096148   
4  sub-01  ses-session1       4   0.575050   0.121707   0.035475  0.052500   

   Fp1_gamma  Fp2_delta  Fp2_theta  Fp2_alpha  Fp2_beta  Fp2_gamma  Fz_delta  \
0   0.039310   0.628390   0.101695   0.032986  0.088670   0.058096  0.489656   
1   0.020225   0.778655   0.055072   0.031630  0.041308   0.035124  0.477474   
2   0.044424   0.469612   0.111364   0.100917  0.117191   0.047921  0.382444   
3   0.030582   0.511625   0.116424   0.053488  0.120168   0.055385  0.466996   
4   0.019073   0.520047   0.111045   0.046061  0.075220   0.034081  0.405714   

   Fz_theta  Fz_alpha   Fz_beta 

In [6]:
META_COLUMNS = [
    "subject",
    "session",
    "window"
]

FEATURE_COLUMNS = [
    col for col in features_df.columns
    if col not in META_COLUMNS
]

X = features_df[FEATURE_COLUMNS]

In [7]:
print("Number of features:", len(FEATURE_COLUMNS))
print("Subjects:", features_df["subject"].nunique())
print("Sessions:", features_df["session"].unique())

Number of features: 15
Subjects: 60
Sessions: <StringArray>
['ses-session1', 'ses-session2', 'ses-session3']
Length: 3, dtype: str


In [8]:
ENROLL_SESSIONS = [
    "ses-session1",
    "ses-session2"
]

VERIFY_SESSION = "ses-session3"

In [9]:
enrollment_df = features_df[
    features_df["session"].isin(ENROLL_SESSIONS)
].copy()

verification_df = features_df[
    features_df["session"] == VERIFY_SESSION
].copy()

print("Enrollment:", enrollment_df.shape)
print("Verification:", verification_df.shape)

Enrollment: (6100, 18)
Verification: (2635, 18)


In [10]:
svm_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "svm",
        SVC(
            kernel="rbf",
            probability=True,
            random_state=42
        )
    )
])

In [11]:
svm_model.fit(
    enrollment_df[FEATURE_COLUMNS],
    enrollment_df["subject"]
)

c:\Users\Sanjeev\OneDrive\Desktop\Brain ID\Model\EEG-Authentication\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('svm', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](60,)","['sub-01','sub-02','sub-03',...,'sub-58','sub-59','sub-60']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](15,)","['Fp1_delta','Fp1_theta','Fp1_alpha',...,'Fz_alpha','Fz_beta','Fz_gamma']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,15
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [12]:
verification_probabilities = svm_model.predict_proba(
    verification_df[FEATURE_COLUMNS]
)

In [13]:
svm_model.classes_

array(['sub-01', 'sub-02', 'sub-03', 'sub-04', 'sub-05', 'sub-06',
       'sub-07', 'sub-08', 'sub-09', 'sub-10', 'sub-11', 'sub-12',
       'sub-13', 'sub-14', 'sub-15', 'sub-16', 'sub-17', 'sub-18',
       'sub-19', 'sub-20', 'sub-21', 'sub-22', 'sub-23', 'sub-24',
       'sub-25', 'sub-26', 'sub-27', 'sub-28', 'sub-29', 'sub-30',
       'sub-31', 'sub-32', 'sub-33', 'sub-34', 'sub-35', 'sub-36',
       'sub-37', 'sub-38', 'sub-39', 'sub-40', 'sub-41', 'sub-42',
       'sub-43', 'sub-44', 'sub-45', 'sub-46', 'sub-47', 'sub-48',
       'sub-49', 'sub-50', 'sub-51', 'sub-52', 'sub-53', 'sub-54',
       'sub-55', 'sub-56', 'sub-57', 'sub-58', 'sub-59', 'sub-60'],
      dtype=object)

In [14]:
print(
    "Number of classes:",
    len(svm_model.classes_)
)

print(
    svm_model.classes_
)

Number of classes: 60
['sub-01' 'sub-02' 'sub-03' 'sub-04' 'sub-05' 'sub-06' 'sub-07' 'sub-08'
 'sub-09' 'sub-10' 'sub-11' 'sub-12' 'sub-13' 'sub-14' 'sub-15' 'sub-16'
 'sub-17' 'sub-18' 'sub-19' 'sub-20' 'sub-21' 'sub-22' 'sub-23' 'sub-24'
 'sub-25' 'sub-26' 'sub-27' 'sub-28' 'sub-29' 'sub-30' 'sub-31' 'sub-32'
 'sub-33' 'sub-34' 'sub-35' 'sub-36' 'sub-37' 'sub-38' 'sub-39' 'sub-40'
 'sub-41' 'sub-42' 'sub-43' 'sub-44' 'sub-45' 'sub-46' 'sub-47' 'sub-48'
 'sub-49' 'sub-50' 'sub-51' 'sub-52' 'sub-53' 'sub-54' 'sub-55' 'sub-56'
 'sub-57' 'sub-58' 'sub-59' 'sub-60']


In [15]:
classes = svm_model.classes_

trial_rows = []

for i, (_, row) in enumerate(
    verification_df.iterrows()
):

    true_subject = row["subject"]

    for class_idx, claimed_subject in enumerate(classes):

        score = verification_probabilities[
            i,
            class_idx
        ]

        genuine = (
            claimed_subject == true_subject
        )

        trial_rows.append({

            "true_subject":
                true_subject,

            "claimed_subject":
                claimed_subject,

            "window":
                row["window"],

            "score":
                score,

            "genuine":
                genuine
        })

trials_df = pd.DataFrame(trial_rows)

In [16]:
print(trials_df.shape)
print(trials_df.head())

(158100, 5)
  true_subject claimed_subject  window     score  genuine
0       sub-01          sub-01       0  0.008768     True
1       sub-01          sub-02       0  0.000247    False
2       sub-01          sub-03       0  0.001039    False
3       sub-01          sub-04       0  0.000243    False
4       sub-01          sub-05       0  0.001948    False


In [17]:
print(
    trials_df["genuine"].value_counts()
)

genuine
False    155465
True       2635
Name: count, dtype: int64


In [18]:
def calculate_far_frr(trials, threshold):

    accepted = trials["score"] >= threshold

    genuine = trials["genuine"]
    impostor = ~genuine

    false_rejects = (
        (~accepted) & genuine
    ).sum()

    genuine_count = genuine.sum()

    false_accepts = (
        accepted & impostor
    ).sum()

    impostor_count = impostor.sum()

    frr = (
        false_rejects / genuine_count
        if genuine_count > 0
        else np.nan
    )

    far = (
        false_accepts / impostor_count
        if impostor_count > 0
        else np.nan
    )

    tar = 1 - frr

    return far, frr, tar

In [19]:
far, frr, tar = calculate_far_frr(
    trials_df,
    threshold=0.5
)

print("FAR:", far)
print("FRR:", frr)
print("TAR:", tar)

FAR: 0.0018139130994114432
FRR: 0.9130929791271347
TAR: 0.08690702087286528


In [20]:
thresholds = np.linspace(
    0,
    1,
    1001
)

results = []

for threshold in thresholds:

    far, frr, tar = calculate_far_frr(
        trials_df,
        threshold
    )

    results.append({
        "threshold": threshold,
        "FAR": far,
        "FRR": frr,
        "TAR": tar
    })

metrics_df = pd.DataFrame(results)

In [21]:
metrics_df["difference"] = (
    metrics_df["FAR"] -
    metrics_df["FRR"]
).abs()

eer_row = metrics_df.loc[
    metrics_df["difference"].idxmin()
]

print(eer_row)

threshold     0.010000
FAR           0.241778
FRR           0.251613
TAR           0.748387
difference    0.009835
Name: 10, dtype: float64


In [22]:
eer = (
    eer_row["FAR"] +
    eer_row["FRR"]
) / 2

print("EER:", eer)

print("EER %:", eer * 100)
print(
    "EER threshold:",
    eer_row["threshold"]
)

EER: 0.24669539767793391
EER %: 24.669539767793392
EER threshold: 0.01


Inspect genuine vs impostor scores

In [23]:
print("Genuine scores:")
print(
    trials_df.loc[
        trials_df["genuine"],
        "score"
    ].describe()
)

print("\nImpostor scores:")
print(
    trials_df.loc[
        ~trials_df["genuine"],
        "score"
    ].describe()
)

Genuine scores:
count    2635.000000
mean        0.139174
std         0.213259
min         0.000346
25%         0.009840
50%         0.038928
75%         0.162320
max         0.979683
Name: score, dtype: float64

Impostor scores:
count    155465.000000
mean          0.014590
std           0.045858
min           0.000018
25%           0.000941
50%           0.002727
75%           0.009522
max           0.943349
Name: score, dtype: float64


In [24]:
print(
    "Top genuine scores:"
)

print(
    trials_df.loc[
        trials_df["genuine"]
    ]
    .sort_values("score", ascending=False)
    [["true_subject", "claimed_subject", "score"]]
    .head(20)
)

print(
    "\nTop impostor scores:"
)

print(
    trials_df.loc[
        ~trials_df["genuine"]
    ]
    .sort_values("score", ascending=False)
    [["true_subject", "claimed_subject", "score"]]
    .head(20)
)

Top genuine scores:
      true_subject claimed_subject     score
76769       sub-30          sub-30  0.979683
77909       sub-30          sub-30  0.978802
77729       sub-30          sub-30  0.973762
77789       sub-30          sub-30  0.969993
77069       sub-30          sub-30  0.961124
75028       sub-29          sub-29  0.961058
79589       sub-30          sub-30  0.959129
77249       sub-30          sub-30  0.957246
72568       sub-29          sub-29  0.955297
78029       sub-30          sub-30  0.946116
96396       sub-37          sub-37  0.942289
35111       sub-12          sub-12  0.942142
76109       sub-30          sub-30  0.940117
79409       sub-30          sub-30  0.939799
78809       sub-30          sub-30  0.939032
76589       sub-30          sub-30  0.931738
77309       sub-30          sub-30  0.929799
78269       sub-30          sub-30  0.927886
76469       sub-30          sub-30  0.921602
78329       sub-30          sub-30  0.919397

Top impostor scores:
       true_s

In [25]:
for threshold in [
    0.001,
    0.005,
    0.01,
    0.02,
    0.05,
    0.10,
    0.20,
    0.50
]:

    far, frr, tar = calculate_far_frr(
        trials_df,
        threshold
    )

    print(
        f"Threshold={threshold:.3f} | "
        f"FAR={far:.4f} | "
        f"FRR={frr:.4f} | "
        f"TAR={tar:.4f}"
    )

Threshold=0.001 | FAR=0.7361 | FRR=0.0247 | TAR=0.9753
Threshold=0.005 | FAR=0.3701 | FRR=0.1400 | TAR=0.8600
Threshold=0.010 | FAR=0.2418 | FRR=0.2516 | TAR=0.7484
Threshold=0.020 | FAR=0.1415 | FRR=0.3666 | TAR=0.6334
Threshold=0.050 | FAR=0.0608 | FRR=0.5514 | TAR=0.4486
Threshold=0.100 | FAR=0.0288 | FRR=0.6698 | TAR=0.3302
Threshold=0.200 | FAR=0.0116 | FRR=0.7776 | TAR=0.2224
Threshold=0.500 | FAR=0.0018 | FRR=0.9131 | TAR=0.0869


05_model_comparison.ipynb

In [26]:
import numpy as np
import pandas as pd

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [27]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y_train = label_encoder.fit_transform(
    enrollment_df["subject"]
)

X_train = enrollment_df[FEATURE_COLUMNS]

X_test = verification_df[FEATURE_COLUMNS]

y_test = label_encoder.transform(
    verification_df["subject"]
)

print("Classes:", len(label_encoder.classes_))

Classes: 60


In [28]:
dt_model = DecisionTreeClassifier(
    random_state=42,
    max_depth=None
)

dt_model.fit(
    X_train,
    y_train
)

,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split among considered features for this split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: splitting may inspect more than ``max_features`` features ifneeded to find a valid split.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples

In [29]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

rf_model.fit(
    X_train,
    y_train
)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_fe

In [30]:
et_model = ExtraTreesClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

et_model.fit(
    X_train,
    y_train
)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls 3 sources of randomness:- the bootstrapping of the samples used when building trees (if ``bootstrap=True``)- the sampling of the features to consider when looking for the best split at each node (if ``max_features < n_features``)- the draw of the splits for each of the `max_features`See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when l

In [31]:
xgb_model = XGBClassifier(
    objective="multi:softprob",
    num_class=60,
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="mlogloss",
    n_jobs=-1
)

xgb_model.fit(
    X_train,
    y_train
)

,"objective objective: typing.Union[str, xgboost.objective.Objective, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn

In [32]:
lgbm_model = LGBMClassifier(
    objective="multiclass",
    num_class=60,
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    verbosity=-1,
    n_jobs=-1
)

lgbm_model.fit(
    X_train,
    y_train
)

,learning_rate,0.05
,n_estimators,300
,objective,'multiclass'
,random_state,42
,n_jobs,-1
,num_class,60
,verbosity,-1
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,subsample_for_bin,200000


In [33]:
def evaluate_model(model, model_name):

    probabilities = model.predict_proba(
        verification_df[FEATURE_COLUMNS]
    )

    classes = model.classes_

    trial_rows = []

    for i, (_, row) in enumerate(
        verification_df.iterrows()
    ):

        true_subject = row["subject"]

        for class_idx, claimed_class in enumerate(classes):

            claimed_subject = label_encoder.inverse_transform(
                [claimed_class]
            )[0]

            score = probabilities[i, class_idx]

            genuine = (
                claimed_subject == true_subject
            )

            trial_rows.append({
                "true_subject": true_subject,
                "claimed_subject": claimed_subject,
                "window": row["window"],
                "score": score,
                "genuine": genuine
            })

    trials = pd.DataFrame(trial_rows)

    thresholds = np.linspace(0, 1, 1001)

    results = []

    for threshold in thresholds:

        far, frr, tar = calculate_far_frr(
            trials,
            threshold
        )

        results.append({
            "threshold": threshold,
            "FAR": far,
            "FRR": frr,
            "TAR": tar
        })

    metrics = pd.DataFrame(results)

    metrics["difference"] = (
        metrics["FAR"] -
        metrics["FRR"]
    ).abs()

    eer_row = metrics.loc[
        metrics["difference"].idxmin()
    ]

    eer = (
        eer_row["FAR"] +
        eer_row["FRR"]
    ) / 2

    print(
        f"{model_name}: "
        f"EER = {eer * 100:.2f}% | "
        f"Threshold = {eer_row['threshold']:.4f}"
    )

    return {
        "model": model_name,
        "EER": eer,
        "threshold": eer_row["threshold"],
        "FAR": eer_row["FAR"],
        "FRR": eer_row["FRR"],
        "TAR": eer_row["TAR"],
        "trials": trials,
        "metrics": metrics
    }

In [34]:
results = []

results.append(
    evaluate_model(
        dt_model,
        "Decision Tree"
    )
)

results.append(
    evaluate_model(
        rf_model,
        "Random Forest"
    )
)

results.append(
    evaluate_model(
        et_model,
        "Extra Trees"
    )
)

results.append(
    evaluate_model(
        xgb_model,
        "XGBoost"
    )
)

results.append(
    evaluate_model(
        lgbm_model,
        "LightGBM"
    )
)

Decision Tree: EER = 44.40% | Threshold = 0.0010
Random Forest: EER = 24.43% | Threshold = 0.0140
Extra Trees: EER = 24.21% | Threshold = 0.0140
XGBoost: EER = 24.93% | Threshold = 0.0040
LightGBM: EER = 27.79% | Threshold = 0.0010


In [35]:
comparison_df = pd.DataFrame([
    {
        "Model": r["model"],
        "EER (%)": r["EER"] * 100,
        "FAR (%)": r["FAR"] * 100,
        "FRR (%)": r["FRR"] * 100,
        "TAR (%)": r["TAR"] * 100,
        "Threshold": r["threshold"]
    }
    for r in results
])

comparison_df.sort_values(
    "EER (%)"
)

,Model,EER (%),FAR (%),FRR (%),TAR (%),Threshold
2,Extra Trees,24.205127,25.601904,22.808349,77.191651,0.014
1,Random Forest,24.425112,25.055157,23.795066,76.204934,0.014
3,XGBoost,24.925868,24.272987,25.578748,74.421252,0.004
4,LightGBM,27.790821,12.887145,42.694497,57.305503,0.001
0,Decision Tree,44.402277,1.480076,87.324478,12.675522,0.001


SHAP Analysis

In [37]:
%pip install shap

^C
Note: you may need to restart the kernel to use updated packages.


In [41]:
import shap

In [43]:
# Get the trained Extra Trees classifier

print(et_model)

ExtraTreesClassifier(class_weight='balanced', n_estimators=300, n_jobs=-1,
                     random_state=42)


In [44]:
# Extract training features
X_train = enrollment_df[FEATURE_COLUMNS].copy()
y_train = enrollment_df["subject"]

# SHAP TreeExplainer
explainer = shap.TreeExplainer(et_model)

# Use a representative sample for SHAP
# This keeps computation manageable
X_shap = X_train.sample(
    n=min(2000, len(X_train)),
    random_state=42
)

print("SHAP samples:", X_shap.shape)

SHAP samples: (2000, 15)


In [ ]:
shap_values = explainer.shap_values(X_shap)

print("SHAP type:", type(shap_values))

if isinstance(shap_values, list):
    print("Number of classes:", len(shap_values))
    print("Shape of first class:", shap_values[0].shape)
else:
    print("SHAP shape:", np.array(shap_values).shape)

In [ ]:
# Convert SHAP values into a single global importance score
if isinstance(shap_values, list):
    # Multiclass:
    # mean absolute SHAP across samples and classes
    shap_importance = np.mean(
        [
            np.abs(class_shap).mean(axis=0)
            for class_shap in shap_values
        ],
        axis=0
    )
else:
    shap_array = np.array(shap_values)

    # Handle possible multiclass dimensions
    if shap_array.ndim == 3:
        shap_importance = np.abs(shap_array).mean(axis=(0, 1))
    else:
        shap_importance = np.abs(shap_array).mean(axis=0)

feature_importance = pd.DataFrame({
    "Feature": FEATURE_COLUMNS,
    "Mean_Absolute_SHAP": shap_importance
}).sort_values(
    "Mean_Absolute_SHAP",
    ascending=False
)

feature_importance